# **Qwen3-4B&Lora**

In [1]:
# set config
CONFIG = dict(
    MODEL_PATH="Qwen/Qwen3-4B-Instruct-2507",
    LEARNING_RATE=1e-3,
    EPOCH=10,
    BATCH_SIZE=16,
    MAX_LENGTH=1024,
    SEED=42,

    LORA_R=16,
    LORA_ALPHA=32,
    LORA_DROPOUT=0.05,
    LORA_TARGET_MODULES=["q_proj", "k_proj", "v_proj", "o_proj"],
)

project_path = "/content/drive/MyDrive/Lectures/2025-26 Spring/CS445/Project"

In [2]:
!pip install -qU torchao

# import libs
import os
import json
import zipfile

import seaborn as sns
from pprint import pprint
import matplotlib.pyplot as plt

import re
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from scipy.stats import spearmanr
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM, get_cosine_schedule_with_warmup
from peft import LoraConfig, get_peft_model, TaskType, set_peft_model_state_dict, load_peft_weights

# set seeds
def seed_all():
    seed = CONFIG["SEED"]
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_all()

# enable optimization and determinism
torch.backends.cudnn.allow_tf32 = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# set theme
sns.set_style("whitegrid")

# mount drive
from google.colab import drive
drive.mount('/content/drive')

# set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# print config
print("\nConfig >")
pprint(CONFIG)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using device: cuda

Config >
{'BATCH_SIZE': 16,
 'EPOCH': 10,
 'LEARNING_RATE': 0.001,
 'LORA_ALPHA': 32,
 'LORA_DROPOUT': 0.05,
 'LORA_R': 16,
 'LORA_TARGET_MODULES': ['q_proj', 'k_proj', 'v_proj', 'o_proj'],
 'MAX_LENGTH': 1024,
 'MODEL_PATH': 'Qwen/Qwen3-4B-Instruct-2507',
 'SEED': 42}


## **Dataset**

In [3]:
# extract dataset
project_path = "/content/drive/MyDrive/Lectures/2025-26 Spring/CS445/Project"
with open(os.path.join(project_path, "train.json"), "r") as train_file: train_df = pd.read_json(train_file).T
with open(os.path.join(project_path, "dev.json"), "r") as validation_file: validation_df = pd.read_json(validation_file).T
with open(os.path.join(project_path, "test.json"), "r") as test_file: test_df = pd.read_json(test_file).T

print(f"{len(train_df)} train examples, {len(validation_df)} validation examples, {len(test_df)} test examples")

2280 train examples, 588 validation examples, 930 test examples


In [4]:
# display example data
homonym = train_df.loc[0, "homonym"]
display(train_df[train_df["homonym"] == homonym])

,homonym,judged_meaning,precontext,sentence,ending,choices,average,stdev,nonsensical,sample_id,example_sentence
0,potential,the difference in electrical charge between tw...,The old machine hummed in the corner of the wo...,The potential couldn't be measured.,She collected a battery reader and looked on e...,"[4, 5, 2, 3, 1]",3.0,1.581139,"[False, False, False, False, False]",1843,The circuit has a high potential difference.
1,potential,the inherent capacity for coming into being,The old machine hummed in the corner of the wo...,The potential couldn't be measured.,She collected a battery reader and looked on e...,"[5, 3, 4, 4, 3]",3.8,0.83666,"[False, False, False, False, False]",1844,The project has great potential for success.
2,potential,the difference in electrical charge between tw...,The old machine hummed in the corner of the wo...,The potential couldn't be measured.,The machine could make such wonderful clothing...,"[2, 1, 4, 3, 1]",2.2,1.30384,"[False, False, False, False, False]",1845,The circuit has a high potential difference.
3,potential,the inherent capacity for coming into being,The old machine hummed in the corner of the wo...,The potential couldn't be measured.,The machine could make such wonderful clothing...,"[4, 5, 5, 3, 5]",4.4,0.894427,"[False, False, False, False, False]",1846,The project has great potential for success.
4,potential,the difference in electrical charge between tw...,The old machine hummed in the corner of the wo...,The potential couldn't be measured.,,"[1, 1, 4, 4, 3]",2.6,1.516575,"[False, False, False, False, False]",1847,The circuit has a high potential difference.
5,potential,the inherent capacity for coming into being,The old machine hummed in the corner of the wo...,The potential couldn't be measured.,,"[5, 4, 3, 4, 5]",4.2,0.83666,"[False, False, False, False, False]",1848,The project has great potential for success.


In [5]:
class AmbiStoryDataset(Dataset):
    system_prompt = """You are an expert linguist analyzing semantic ambiguity by rating the plausibility of a specific word sense for a given homonym, its sense, precontext, ambiguous ending.
Precontext consists of three sentences that ground the story. Ambiguous sentence contains the homonym that causes it to have two different plausible interpretations.
Output only a decimal numerical score between 1 and 5, where 1 means the word sense is completely implausible and 5 means it is highly plausible.
"""
    user_prompt = """Homonym: {homonym}
Sense: {judged_meaning}
Example Usage: {example_sentence}

Precontext: {precontext}
Ambiguous Ending: {ambigious_ending}
Score: """

    def __init__(self, data_df, is_training_split, tokenizer):
        super().__init__()
        self.tokenizer = tokenizer
        self.data_df = data_df.copy()
        self.is_training_split = is_training_split

        # generate prompts
        self.data_df["user_prompt"] = self.data_df.apply(self.generate_prompt, axis=1)

        # generate labels (-1 for test examples)
        self.data_df[self.data_df[["average", "stdev"]] == "(???)"] = -1
        self.data_df["score"] = self.data_df["average"].astype(float)
        self.data_df["std"] = self.data_df["stdev"].astype(float)

    def generate_prompt(self, row):
        homonym = row["homonym"]
        judged_meaning = row["judged_meaning"]
        example_sentence = row["example_sentence"]
        precontext = row["precontext"]
        ambigious_ending = row["sentence"] + (" " + row["ending"] if row["ending"] else "")

        return self.user_prompt.format(
            homonym=homonym,
            judged_meaning=judged_meaning,
            example_sentence=example_sentence,
            precontext=precontext,
            ambigious_ending=ambigious_ending
        )

    def __len__(self):
        return len(self.data_df)

    def __getitem__(self, idx):
        messages = [
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": self.data_df.loc[idx, "user_prompt"]}
        ]

        if self.is_training_split:
            messages.append({"role": "assistant", "content": str(self.data_df.loc[idx, "score"])})

        prompt = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=not self.is_training_split
        )

        return {
            "prompt": prompt,
            "score": self.data_df.loc[idx, "score"],
            "std": self.data_df.loc[idx, "std"]
        }

In [6]:
class DataCollator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer
        self.max_length = CONFIG["MAX_LENGTH"]

    def __call__(self, batch):
        prompts = [item["prompt"] for item in batch]
        scores = [item["score"] for item in batch]
        stds = [item["std"] for item in batch]

        tokenized_prompt = self.tokenizer(
            prompts,
            truncation=True,
            max_length=self.max_length,
            padding=True,
            return_tensors="pt"
        )

        input_ids = tokenized_prompt["input_ids"]
        attention_mask = tokenized_prompt["attention_mask"]
        labels = input_ids.clone()
        labels[attention_mask == 0] = -100

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
            "score": torch.FloatTensor(scores),
            "std": torch.FloatTensor(stds)
        }

In [7]:
# define dataset and dataloader
tokenizer = AutoTokenizer.from_pretrained(CONFIG["MODEL_PATH"])
tokenizer.padding_side = "left"
train_dataset = AmbiStoryDataset(train_df, True, tokenizer)
validation_dataset = AmbiStoryDataset(validation_df, False, tokenizer)
test_dataset = AmbiStoryDataset(test_df, False, tokenizer)

collate_fn = DataCollator(tokenizer)
train_dataloader = DataLoader(train_dataset, batch_size=CONFIG["BATCH_SIZE"], shuffle=True, collate_fn=collate_fn, pin_memory=True)
validation_dataloader = DataLoader(validation_dataset, batch_size=CONFIG["BATCH_SIZE"], shuffle=False, collate_fn=collate_fn, pin_memory=True)
test_dataloader = DataLoader(test_dataset, batch_size=CONFIG["BATCH_SIZE"], shuffle=False, collate_fn=collate_fn, pin_memory=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [8]:
# show example
row = train_dataset[0]
print(f"Prompt:\n{row['prompt']}")
print(f"Score: {row['score']}")
print(f"Std: {row['std']}")

Prompt:
<|im_start|>system
You are an expert linguist analyzing semantic ambiguity by rating the plausibility of a specific word sense for a given homonym, its sense, precontext, ambiguous ending.
Precontext consists of three sentences that ground the story. Ambiguous sentence contains the homonym that causes it to have two different plausible interpretations.
Output only a decimal numerical score between 1 and 5, where 1 means the word sense is completely implausible and 5 means it is highly plausible.
<|im_end|>
<|im_start|>user
Homonym: potential
Sense: the difference in electrical charge between two points in a circuit expressed in volts
Example Usage: The circuit has a high potential difference.

Precontext: The old machine hummed in the corner of the workshop. Clara examined its dusty dials with a furrowed brow. She wondered if it could be brought back to life.
Ambiguous Ending: The potential couldn't be measured. She collected a battery reader and looked on earnestly, willing so

## **Training**

In [9]:
class LLM(torch.nn.Module):
    def __init__(self):
        super().__init__()

        self.model_path = CONFIG["MODEL_PATH"]
        lora_config = LoraConfig(
            r=CONFIG["LORA_R"],
            lora_alpha=CONFIG["LORA_ALPHA"],
            target_modules=CONFIG["LORA_TARGET_MODULES"],
            lora_dropout=CONFIG["LORA_DROPOUT"],
            task_type=TaskType.CAUSAL_LM
        )

        self.model = AutoModelForCausalLM.from_pretrained(self.model_path)
        self.model = get_peft_model(self.model, lora_config)

    def forward(self, **kwargs):
        return self.model(**kwargs)

In [10]:
def train(model, dataloader, optimizer, scheduler):
    model.to(device)
    model.train()

    train_loss = 0
    progress_bar = tqdm(dataloader, desc="Training  ")
    for batch_index, batch in enumerate(progress_bar):
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)

        optimizer.zero_grad()
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        loss = outputs.loss
        loss.backward()
        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)

        optimizer.step()
        scheduler.step()

        train_loss += loss.item()*input_ids.size(0)
        progress_bar.set_postfix({"loss": f"{loss.item():.4f}", "grad_norm": f"{grad_norm:.4f}"})

    train_loss /= len(dataloader.dataset)
    return train_loss

In [11]:
def extract_scores(decoded_outputs):
    scores = list()
    for decoded_output in decoded_outputs:
        text = decoded_output.strip()
        score_match = re.search(r"([1-4](\.\d+)?|5(\.0+)?)", text)
        if score_match:
            score = float(score_match.group(0))
            scores.append(score)
        else:
            scores.append(-1)
    return scores

In [12]:
def test(model, dataloader, validation):
    model.to(device)
    model.eval()

    all_preds = []
    all_scores = []
    all_stds = []
    progress_bar = tqdm(dataloader, desc="Validation" if validation is True else "Testing   ")
    with torch.no_grad():
        for batch_index, batch in enumerate(progress_bar):
            input_ids = batch["input_ids"].to(device, non_blocking=True)
            attention_mask = batch["attention_mask"].to(device, non_blocking=True)
            scores = batch["score"].numpy()
            stds = batch["std"].numpy()

            generated_ids = model.model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=5,
                pad_token_id=tokenizer.pad_token_id,
                do_sample=False,
                use_cache=True
            )

            new_tokens = generated_ids[:, input_ids.shape[1]:]
            decoded_outputs = tokenizer.batch_decode(new_tokens, skip_special_tokens=True)
            pred_scores = extract_scores(decoded_outputs)

            all_preds.extend(pred_scores)
            all_scores.extend(scores)
            all_stds.extend(stds)

    if validation:
        all_preds = np.array(all_preds)
        all_scores = np.array(all_scores)
        all_stds = np.array(all_stds)

        spearman_corr, _ = spearmanr(all_preds, all_scores)
        mae = np.mean(np.abs(all_preds - all_scores))
        rmse = np.sqrt(np.mean((all_preds - all_scores)**2))
        within_std = np.abs(all_preds - all_scores) <= np.maximum(all_stds, 1.0)
        acc_within_std = np.mean(within_std)*100
        scores = {
            "spearman_corr": spearman_corr,
            "acc_within_std": acc_within_std,
            "mae": mae,
            "rmse": rmse
        }

        return scores, all_preds

    else:
        zip_filename = "predictions.zip"
        jsonl_filename = "predictions.jsonl"
        with open(jsonl_filename, "w") as f:
            for pred_index, pred in enumerate(all_preds):
                submission_dict = {"id": str(pred_index), "prediction": pred}
                f.write(json.dumps(submission_dict) + "\n")

        with zipfile.ZipFile(zip_filename, "w", zipfile.ZIP_DEFLATED) as zipf:
            zipf.write(jsonl_filename, arcname=jsonl_filename)

        return all_preds

In [13]:
def optimize(train_dataloader, validation_dataloader):
    seed_all()

    lr = CONFIG["LEARNING_RATE"]
    max_epochs = CONFIG["EPOCH"]
    train_weight_path = os.path.join("/content/train")
    train_history_path = os.path.join("/content/train/history.json")
    history = dict(train_loss=[], spearman=[], acc_std=[], mae=[], rmse=[], epoch=0, lr=lr, output="", config=CONFIG)


    print(f"Learning Rate is set to {lr}")
    history["output"] += f"Learning Rate is set to {lr}\n"


    print(f"\n{30*'-'} Training {30*'-'}")
    history["output"] += f"\n{30*'-'} Training {30*'-'}\n"

    model = LLM()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    total_steps = max_epochs*len(train_dataloader)
    warmup_steps = int(0.1*total_steps)
    scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)

    best_spearman = float("-inf")
    best_epoch = 0
    for epoch in range(max_epochs):
        print(f"Epoch: {epoch+1}/{max_epochs} > ")
        history["output"] += f"Epoch: {epoch+1}/{max_epochs} > \n"

        train_loss = train(model, train_dataloader, optimizer, scheduler)
        scores, _ = test(model, validation_dataloader, True)

        val_spearman = scores["spearman_corr"]
        val_acc_std = scores["acc_within_std"]
        val_mae = scores["mae"]
        val_rmse = scores["rmse"]

        print(f"\tResults > Train Loss: {train_loss:.4f}, Spearman: {val_spearman:.4f}, Acc-STD: {val_acc_std:.2f}%, MAE: {val_mae:.4f}, RMSE: {val_rmse:.4f}\n")
        history["output"] += f"\tTrain Loss: {train_loss:.4f}, Spearman: {val_spearman:.4f}, Acc-STD: {val_acc_std:.2f}%, MAE: {val_mae:.4f}, RMSE: {val_rmse:.4f}\n\n"

        history["train_loss"].append(train_loss)
        history["spearman"].append(val_spearman)
        history["acc_std"].append(val_acc_std)
        history["mae"].append(val_mae)
        history["rmse"].append(val_rmse)

        if val_spearman > best_spearman:
            best_spearman = val_spearman
            best_epoch = epoch + 1
            history["epoch"] = best_epoch

            os.makedirs(os.path.dirname(train_weight_path), exist_ok=True)
            model.model.save_pretrained(train_weight_path)

        with open(train_history_path, "w") as history_file:
            json.dump(history, history_file)

    print(f"{30*'-'} Training {30*'-'}")
    history["output"] += f"{30*'-'} Training {30*'-'}\n"


    print(f"Best Epoch is {best_epoch}")
    history["output"] += f"\nBest Epoch is {best_epoch}\n"


    with open(train_history_path, "w") as history_file:
        json.dump(history, history_file)

    best_peft_weights = load_peft_weights(train_weight_path)
    set_peft_model_state_dict(model.model, best_peft_weights)
    model.to(device)

    return model, history

In [14]:
# train model
model, history = optimize(train_dataloader, validation_dataloader)

Learning Rate is set to 0.001

------------------------------ Training ------------------------------


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Epoch: 1/10 > 


Validation: 100%|██████████| 37/37 [00:08<00:00,  4.55it/s]


	Results > Train Loss: 1.2163, Spearman: 0.4176, Acc-STD: 61.90%, MAE: 1.1645, RMSE: 1.5116

Epoch: 2/10 > 


...

	Results > Train Loss: 0.1243, Spearman: 0.6836, Acc-STD: 81.29%, MAE: 0.7031, RMSE: 0.9077

Epoch: 9/10 > 


Validation: 100%|██████████| 37/37 [00:08<00:00,  4.59it/s]


	Results > Train Loss: 0.1215, Spearman: 0.6916, Acc-STD: 82.48%, MAE: 0.6824, RMSE: 0.8734

Epoch: 10/10 > 


Validation: 100%|██████████| 37/37 [00:08<00:00,  4.59it/s]


	Results > Train Loss: 0.1193, Spearman: 0.6949, Acc-STD: 82.48%, MAE: 0.6806, RMSE: 0.8687

------------------------------ Training ------------------------------
Best Epoch is 10


## **Results**

In [15]:
# test results
predictions = test(model, test_dataloader, False)

print()
print(f"Example Predictions > {predictions[:10]}")
print(f"Failed Predictions > {np.sum(predictions == -1)}")

Testing   : 100%|██████████| 59/59 [00:12<00:00,  4.61it/s]


Example Predictions > [5.0, 2.2, 4.2, 3.8, 4.6, 3.2, 3.4, 2.8, 1.8, 2.4]
Failed Predictions > 0
